# Makemore bigram in the browser

This notebook runs locally in JupyterLite/Pyodide using the bundled `torchlite` compatibility wheel.

In [ ]:
%pip install -q torchlite

In [ ]:
import torch
import torch.nn.functional as F

print('torchlite', torch.__version__)

In [ ]:
words = open('data/names.txt').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

xs, ys = [], []
for word in words:
    characters = ['.'] + list(word) + ['.']
    for first, second in zip(characters, characters[1:]):
        xs.append(stoi[first])
        ys.append(stoi[second])

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
classes = len(stoi)
print(f'{num} training bigrams, {classes} classes')

In [ ]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((classes, classes), generator=g, requires_grad=True)
losses = []
xenc = F.one_hot(xs, num_classes=classes).float()  # constant: reuse across steps

for step in range(50):
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W ** 2).mean()
    losses.append(loss.item())
    W.grad = None
    loss.backward()
    W.data += -20 * W.grad

print(f'loss: {losses[0]:.4f} -> {losses[-1]:.4f}')
assert losses[-1] < losses[0]

In [ ]:
samples = []
sample_generator = torch.Generator().manual_seed(2147483647)
for _ in range(5):
    out, ix = [], 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=classes).float()
        logits = xenc @ W
        probabilities = logits.exp()
        probabilities = probabilities / probabilities.sum(1, keepdims=True)
        ix = torch.multinomial(probabilities[0], 1, replacement=True, generator=sample_generator).item()
        out.append(itos[ix])
        if ix == 0 or len(out) >= 20:
            break
    samples.append(''.join(out))
samples